# Send a Message to confluent-kafka Topic via Python

Note:
This configuration is tied to a free-tier Confluent Cloud account used for learning only. It is not meant for production. Sensitive credentials should be kept private and never exposed in public repositories.

In [ ]:
# Required connection configs for Kafka producer, consumer, and admin
bootstrap.servers=pkc-n3603.us-central1.gcp.confluent.cloud:9092
security.protocol=SASL_SSL
sasl.mechanisms=PLAIN
sasl.username=ZTQUCSMEJBUY7PLK
sasl.password=cfltbNQTvE8hHOVo9eOV42+Vi42hiq5ScPxy6xrO30lO3HZQ2yvv2+WtYmgZv/uA

# Best practice for higher availability in librdkafka clients prior to 1.7
session.timeout.ms=45000

client.id=ccloud-python-client-c6482e31-ce2e-468d-affc-04ad676b8bda


In [6]:
!pip install confluent-Kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 42.1 MB/s eta 0:00:00


In [12]:
import pandas as pd
import json

csv_file = 'customer_data/first_100_customers.csv'

df = pd.read_csv(csv_file)
df.count()

,0
customer_id,99
name,99
city,99
state,99
country,99
registration_date,99
is_active,99


In [8]:
# Create a json file for customer data
json_records = df.to_dict(orient='records')

json_file = 'customers.json'

with open(json_file, 'w') as file:
  json.dump(json_records, file, indent=4)
print('File converted to JSON')

File converted to JSON


In [10]:
from confluent_kafka import Producer
import json
import time

# The configuration
kakfa_conf = {
    "bootstrap.servers":"pkc-n3603.us-central1.gcp.confluent.cloud:9092",
    "security.protocol":"SASL_SSL",
    "sasl.mechanisms":"PLAIN",
    "sasl.username":"ZTQUCSMEJBUY7PLK",
    "sasl.password":"cfltbNQTvE8hHOVo9eOV42+Vi42hiq5ScPxy6xrO30lO3HZQ2yvv2+WtYmgZv/uA",
    "session.timeout.ms":"45000",
    "client.id":"ccloud-python-client-c6482e31-ce2e-468d-affc-04ad676b8bda"
}

# creating a producer
producer = Producer(kakfa_conf)


In [4]:
# Topic inside the confluent cluster or name of the cluster 'cluster_0' --> this vary in different project
topic = 'ecommerce'

# Open the customers.json and store it to customers_data
with open('customers.json', 'r') as file:
  customers_data = json.load(file)

# Take the first value in the customers_data
firstValue = customers_data[0]
key = firstValue['customer_id']

print(key,firstValue)

0 {'customer_id': 0, 'name': 'Customer_0', 'city': 'Pune', 'state': 'Maharashtra', 'country': 'India', 'registration_date': '2023-06-29', 'is_active': False}


In [ ]:
# Convert the key and firstValue to bytes first
bytesValue = str(firstValue).encode('utf-8')
bytesKey = str(key).encode('utf-8')

In [ ]:
# Send the first message to Confluent or send only one message to Confluent
producer.produce(topic, key= bytesKey, value = bytesValue)

In [11]:
# Send Multiple messages to kafka Cluster in Confluent cluster or topic
def delivery_status(err, msg):
  if(err):
    print(f'Message Delivery Failed: {err}')
  else:
    print(f'Message Delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}')



for record in customers_data:
  try:
    message_value = json.dumps(record)
    message_key = str(record['customer_id']).encode('utf-8')

    producer.produce(topic, key=message_key, value=message_value, callback= delivery_status)
    producer.poll(1)
  except Exception as e:
    print(f'Error sending messages: {e}')

producer.flush()
print('Messages send to kafka successfully')


Message Delivered to ecommerce [2] at offset 2
Message Delivered to ecommerce [2] at offset 3
Message Delivered to ecommerce [1] at offset 0
Message Delivered to ecommerce [1] at offset 1
Message Delivered to ecommerce [1] at offset 2
Message Delivered to ecommerce [1] at offset 3
Message Delivered to ecommerce [1] at offset 4
Message Delivered to ecommerce [0] at offset 0
Message Delivered to ecommerce [2] at offset 4
Message Delivered to ecommerce [0] at offset 1
Message Delivered to ecommerce [0] at offset 2
Message Delivered to ecommerce [0] at offset 3
Message Delivered to ecommerce [0] at offset 4
Message Delivered to ecommerce [2] at offset 5
Message Delivered to ecommerce [0] at offset 5
Message Delivered to ecommerce [1] at offset 5
Message Delivered to ecommerce [0] at offset 6
Message Delivered to ecommerce [2] at offset 6
Message Delivered to ecommerce [0] at offset 7
Message Delivered to ecommerce [1] at offset 6
Message Delivered to ecommerce [0] at offset 8
Message Deliv